# 01 - M1 gate on the real data
Run after `git pull` in /content/csm-clip (or re-clone). CPU is fine; no GPU needed.
Checks: held-out generator absent from train/val, splits balanced, manifests match the pinned dataset revision, controlled protocol applies exactly one JPEG q90.

In [ ]:
%cd /content/csm-clip
!git pull -q
!pip install -q -r requirements.txt
!python -m pytest -q

In [ ]:
# Load the dataset at the revision pinned in the manifests
import numpy as np
from datasets import load_dataset
from src.data import load_manifest, audit_manifest
FOLDS = ['Midjourney', 'BigGAN', 'ADM']
rev = load_manifest('Midjourney')['revision']
ds = load_dataset('TheKernel01/Tiny-GenImage', revision=rev)
names = ds['train'].features['generator'].names
train_gen = np.asarray(ds['train']['generator']); val_gen = np.asarray(ds['validation']['generator'])
print('revision', rev, '\nnames', names)

In [ ]:
# Gate 1+2: leakage, balance, test 500/500 - fails loudly on any violation
for f in FOLDS:
    m = load_manifest(f)
    assert m['revision'] == rev
    print(f, audit_manifest(m, train_gen, val_gen, names))
# determinism: manifests are fixed files, so re-loading gives identical lists
assert load_manifest('ADM')['train'] == load_manifest('ADM')['train']
print('M1 gate: leakage/balance OK')

In [ ]:
# Gate 3: controlled protocol on real images from every class: one JPEG, 256 px, q90 tables
import io
from PIL import Image
from src.data import controlled_pre, preprocess
rows = []
for g in range(len(names)):
    idx = np.where(train_gen == g)[0][:5]
    for i in idx:
        im = ds['train'][int(i)]['image']
        out = controlled_pre(im)
        assert out.size == (256, 256)
        x = preprocess(im, 'controlled'); assert tuple(x.shape) == (3, 336, 336)
        rows.append((names[g], im.format, im.size))
print(rows[:3], '... checked', len(rows), 'images across', len(set(r[0] for r in rows)), 'classes')
print('controlled protocol OK')